# RAIS - SQL

In [52]:
import basedosdados as bd
import pandas as pd
import openpyxl

RAIS e Caged


Os dados da RAIS (Relação Anual de Informações Sociais) e do CAGED (Cadastro Geral de Empregados e Desempregados) são registros administrativos sobre o mercado de trabalho formal brasileiro, controlados pelo Ministério do Trabalho e Emprego (MTE).

In [53]:
BILLING_ID = "microdados-caged-510923"
ANO_INI, ANO_FIM = 2000, 2014

Precisei criar um ID projeto no Google Cloud 
E instituir os anos que preciso dos dados

O programa que será analisado tem datamento principal entre (2008-2013), mas minhas inferências buscam entender o comportamento das variavéis antes e depois da instituição do programa

In [54]:
UF_TESTE = None

None faz com sirva para todo o Brasil

In [55]:
if UF_TESTE:
    FILTRO_UF = f"AND d.sigla_uf = '{UF_TESTE}'"
else:
    FILTRO_UF = ""

In [56]:
def rodar(query):
    df = bd.read_sql(query=query, billing_project_id=BILLING_ID)
    return df

In [57]:
tabela = "br_me_rais.microdados_vinculos"
dataset, tab = tabela.split(".")

In [58]:
query = f"""
SELECT column_name, data_type
FROM `basedosdados.{dataset}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = '{tab}'
"""

query de exploração --> Ver quais colunas existem na RAIS e escolher as que você usaria.

In [59]:
pd.set_option("display.max_rows", None)

colunas = rodar(query)
colunas

Downloading: 100%|██████████|


,column_name,data_type
0,ano,INT64
1,sigla_uf,STRING
2,id_municipio,STRING
3,tipo_vinculo,STRING
4,vinculo_ativo_3112,STRING
5,tipo_admissao,STRING
6,mes_admissao,INT64
7,mes_desligamento,INT64
8,motivo_desligamento,STRING
9,causa_desligamento_1,STRING


In [65]:
query_rais = f"""
SELECT
    d.ano,
    d.sigla_uf,
    d.id_municipio,
    d.subsetor_ibge,
    COUNT(*) AS emprego_3112,                                                  -- nº de vínculos ativos em 31/12
    AVG(IF(CAST(d.sexo AS STRING) = '1', 1, 0)) AS prop_homens,   -- 1 = masculino
    AVG(d.idade) AS idade_media,                                               -- idade média
    AVG(IF(CAST(d.sexo AS STRING) = '2', 1, 0)) AS prop_mulheres,             -- 2 = feminino
    AVG(IF(CAST(d.raca_cor AS STRING) = '4', 1, 0)) AS prop_preta,            -- 4 = preta
    AVG(IF(CAST(d.raca_cor AS STRING) = '8', 1, 0)) AS prop_parda,            -- 8 = parda
    AVG(IF(CAST(d.raca_cor AS STRING) IN ('4', '8'), 1, 0)) AS prop_negra,    -- 4 + 8 = pretos e pardos
    AVG(d.valor_salario_contratual) AS valor_salario_contratual,              -- salário do contrato
    AVG(d.valor_remuneracao_media) AS valor_remuneracao_media,                -- remuneração média mensal
    AVG(d.valor_remuneracao_dezembro) AS valor_remuneracao_dezembro           -- remuneração de dezembro
FROM `basedosdados.{dataset}.{tab}` AS d
WHERE d.ano BETWEEN {ANO_INI} AND {ANO_FIM}       -- período escolhido
  AND CAST(d.vinculo_ativo_3112 AS STRING) = '1'  -- só vínculos ativos em 31/12
  {FILTRO_UF}                                     -- filtro de UF (se houver)
GROUP BY d.ano, d.sigla_uf, d.id_municipio, d.subsetor_ibge
ORDER BY d.id_municipio, d.ano, d.subsetor_ibge
"""

In [68]:
rais = rodar(query_rais)
rais.head(10)

Downloading: 100%|██████████|


,ano,sigla_uf,id_municipio,subsetor_ibge,emprego_3112,prop_homens,idade_media,prop_mulheres,prop_preta,prop_parda,prop_negra,valor_salario_contratual,valor_remuneracao_media,valor_remuneracao_dezembro
0,2001,IGNORADO,None,None,39,0.589744,41.230769,0.410256,0.0,0.0,0.0,NaN,807.853077,833.025897
1,2002,IGNORADO,None,None,54,0.500000,37.870370,0.500000,0.0,0.0,0.0,NaN,802.830741,782.951481
2,2003,IGNORADO,None,None,2,0.500000,25.500000,0.500000,0.0,0.0,0.0,NaN,451.735000,569.880000
3,2004,IGNORADO,None,None,1,0.000000,28.000000,1.000000,0.0,0.0,0.0,NaN,403.320000,413.320000
4,2005,IGNORADO,None,None,5,0.400000,35.400000,0.600000,0.0,0.0,0.0,NaN,1231.606000,652.622000
5,2006,IGNORADO,None,None,6,0.333333,33.500000,0.666667,0.0,0.0,0.0,NaN,1120.258333,1148.476667
6,2007,IGNORADO,None,None,4,0.750000,39.750000,0.250000,0.0,0.0,0.0,NaN,1822.425000,2056.255000
7,2000,RO,1100015,None,1196,0.567726,31.193144,0.432274,0.0,0.0,0.0,NaN,287.408094,290.064749
8,2001,RO,1100015,None,1185,0.578903,31.657384,0.421097,0.0,0.0,0.0,NaN,330.574304,323.114203
9,2002,RO,1100015,None,1440,0.578472,32.215972,0.421528,0.0,0.0,0.0,NaN,438.469896,409.444174


In [67]:
rais.to_excel("rais_empr_remuner_2000_2014.xlsx", index=False)